# MLP Bootcamp Day 6: Feature Engineering Hands-On
*Recording of Day 5 Teaching Session: https://drive.google.com/file/d/1RCtlzQ0IvRSRDcirhh1MbRzSMtoLx8Xv/view?usp=sharing*
### Dataset:
`train.csv`: https://drive.google.com/file/d/1zIETXSnQo6QX94vSiXOa4yOY_flHYZBU/view?usp=sharing
<br>
`test.csv`: https://drive.google.com/file/d/18x3wEb-_n0ollIR_ftJ_Wq4dyl3Vzf81/view?usp=sharing

**Q1.** Load the `train.csv` dataset. Split it into a training set (75%) and a validation set (25%) using `train_test_split`. Use `random_state=42` and stratify by the `sentiment` column. What is the shape of the training feature matrix?

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('train.csv')
df.head()

,id,review_date,title,review_text,product_category,user_country,verified_purchase,price,discount,rating,helpful_votes,total_votes,sentiment
0,0,2022-01-31,NaN,VERY BAD QUALITY AND DISAPPOINTING EXPERIENCE....,Home,UK,True,69.44,0.30,2,5,6,negative
1,1,2022-12-30,Bad Purchase,Excellent performance and great value. Rating:...,Books,UK,True,40.69,0.49,3,3,6,neutral
2,2,2021-05-10,Best Buy,Very bad quality and disappointing experience....,Books,Australia,True,40.52,0.38,4,1,5,positive
3,3,2022-07-18,Okay Item,Very bad quality and disappointing experi...,Electronics,Australia,True,41.96,0.25,3,9,16,neutral
4,4,2022-02-04,NaN,THIS PRODUCT IS AMAZING AND WORKS PERFECTLY. R...,Clothing,Canada,False,93.21,0.12,5,6,11,positive


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(df.drop('sentiment', axis=1), df['sentiment'], test_size=0.25, random_state=42)

**Q2.** Identify all numerical columns and categorical columns in the training set (excluding the target `sentiment` and the `id` column). Using `SimpleImputer`, fill missing values in numerical columns with the **median** and categorical columns with the **most frequent** value. Fit the imputer on the training set only and transform both train and validation sets. What is the total sum of null values remaining in the validation set?

In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np

categorical_cols = ['review_date', 'title', 'review_text', 'product_category', 'user_country', 'verified_purchase']
numerical_cols = ['price', 'discount', 'rating', 'helpful_votes', 'total_votes']

imputer_num = SimpleImputer(strategy='median')
imputer_cat = SimpleImputer(strategy='most_frequent')

imputer_num.fit(x_train[numerical_cols])
imputer_cat.fit(x_train[categorical_cols])

x_train_imputed = x_train.copy()
x_train_imputed[numerical_cols] = imputer_num.transform(x_train[numerical_cols])
x_train_imputed[categorical_cols] = imputer_cat.transform(x_train[categorical_cols])

x_test_imputed = x_test.copy()
x_test_imputed[numerical_cols] = imputer_num.transform(x_test[numerical_cols])
x_test_imputed[categorical_cols] = imputer_cat.transform(x_test[categorical_cols])

null_sum = x_test_imputed.isnull().sum().sum()
null_sum_train = x_train_imputed.isnull().sum().sum()
print(f"\nTotal sum of null values remaining in validation set: {null_sum}")
print(f"Total sum of null values remaining in training set: {null_sum_train}")


Total sum of null values remaining in validation set: 0
Total sum of null values remaining in training set: 0


**Q3.** The `review_date` column is currently a string. Convert it to a datetime object. Create a new feature `review_year` and `review_month`. After transformation, drop the original `review_date` column. What is the average `rating` for reviews written in the year 2022 in the training set?

In [ ]:
x_train_imputed['review_date'] = pd.to_datetime(x_train_imputed['review_date'])
x_test_imputed['review_date'] = pd.to_datetime(x_test_imputed['review_date'])

x_train_imputed['review_year'] = x_train_imputed['review_date'].dt.year
x_train_imputed['review_month'] = x_train_imputed['review_date'].dt.month

x_test_imputed['review_year'] = x_test_imputed['review_date'].dt.year
x_test_imputed['review_month'] = x_test_imputed['review_date'].dt.month

x_train_imputed.drop(columns=['review_date'], inplace=True)
x_test_imputed.drop(columns=['review_date'], inplace=True)

avg_rating_2022 = x_train_imputed[x_train_imputed['review_year'] == 2022]['rating'].mean()
print(f"Average rating for reviews in 2022 (training set): {avg_rating_2022:.4f}")

Average rating for reviews in 2022 (training set): 3.4371


**Q4.** **Interaction Feature:** Create a new feature `effective_price` which is calculated as $price \times (1 - discount)$. If the result is NaN due to missing price/discount (before imputation), fill it with 0. What is the maximum `effective_price` in the training set?

In [ ]:
x_train_imputed['effective_price'] = x_train_imputed['price'] * (1 - x_train_imputed['discount'])
x_test_imputed['effective_price'] = x_test_imputed['price'] * (1 - x_test_imputed['discount'])

x_train_imputed['effective_price'] = x_train_imputed['effective_price'].fillna(0)
x_test_imputed['effective_price'] = x_test_imputed['effective_price'].fillna(0)

max_effective_price = x_train_imputed['effective_price'].max()
print(f"Maximum effective_price in the training set: {max_effective_price:.4f}")

Maximum effective_price in the training set: 659.2320


**Q5.** **Text Length Feature:** Create a feature `review_word_count` by counting the number of words in the `review_text` column. Handle any potential NaNs in `review_text` by replacing them with an empty string before counting. What is the Pearson correlation coefficient (rounded to 4 decimal places) between `review_word_count` and `rating`?

In [ ]:
x_train_imputed['review_word_count'] = x_train_imputed['review_text'].fillna('').apply(lambda x: len(x.split()))
x_test_imputed['review_word_count'] = x_test_imputed['review_text'].fillna('').apply(lambda x: len(x.split()))

pearson_corr = x_train_imputed['review_word_count'].corr(x_train_imputed['rating'])
print(f"Pearson correlation between review_word_count and rating: {pearson_corr:.4f}")

Pearson correlation between review_word_count and rating: 0.0091


**Q6.** **Binary Flagging:** Create a binary feature `is_edited` which is 1 if the `review_text` contains any special characters like "!!!" or "???" and 0 otherwise. What percentage of the training data is marked as "edited"?

In [ ]:
x_train_imputed['is_edited'] = x_train_imputed['review_text'].apply(lambda x: 1 if '!!!' in str(x) or '???' in str(x) else 0)
x_test_imputed['is_edited'] = x_test_imputed['review_text'].apply(lambda x: 1 if '!!!' in str(x) or '???' in str(x) else 0)

edited_percentage = x_train_imputed['is_edited'].mean() * 100
print(f"Percentage of training data marked as 'edited': {edited_percentage:.4f}%")

Percentage of training data marked as 'edited': 31.2667%


**Q7.** **Ratio Feature:** Create a feature `helpfulness_ratio` by dividing `helpful_votes` by `total_votes`. To avoid division by zero, add 1 to the denominator ($helpful\_votes / (total\_votes + 1)$). What is the mean value of this ratio in the training set?

In [ ]:
x_train_imputed['helpfulness_ratio'] = x_train_imputed['helpful_votes'] / (x_train_imputed['total_votes'] + 1)
x_test_imputed['helpfulness_ratio'] = x_test_imputed['helpful_votes'] / (x_test_imputed['total_votes'] + 1)

mean_helpfulness_ratio = x_train_imputed['helpfulness_ratio'].mean()
print(f"Mean helpfulness_ratio in the training set: {mean_helpfulness_ratio:.4f}")

Mean helpfulness_ratio in the training set: 0.4167


**Q8.** Use `LabelEncoder` to transform the target variable `sentiment` into numerical labels. Mapping should be alphabetical (negative=0, neutral=1, positive=2). What is the count of the label '1' (neutral) in your training target array?

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

print(f"Classes: {le.classes_}")
neutral_count = (y_train_encoded == 1).sum()
print(f"Count of label '1' (neutral) in training target array: {neutral_count}")

Classes: ['negative' 'neutral' 'positive']
Count of label '1' (neutral) in training target array: 3031


**Q9.** Apply `OneHotEncoder` to the `product_category` and `user_country` columns. Use `sparse=False` and `handle_unknown='ignore'`. How many new columns were created from these two features?

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe_cols = ['product_category', 'user_country']

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe.fit(x_train_imputed[ohe_cols])

x_train_ohe = ohe.transform(x_train_imputed[ohe_cols])
x_test_ohe = ohe.transform(x_test_imputed[ohe_cols])

num_new_cols = x_train_ohe.shape[1]
print(f"Number of new columns created from OneHotEncoding: {num_new_cols}")
print(f"Categories per feature: { {col: len(cats) for col, cats in zip(ohe_cols, ohe.categories_)} }")

Number of new columns created from OneHotEncoding: 14
Categories per feature: {'product_category': 7, 'user_country': 7}


**Q10.** **Standardization:** Apply `StandardScaler` to all numerical features (including the new ones created in Q4, Q5, and Q7). After scaling, what is the standard deviation of the `price` column in the training set?

In [ ]:
from sklearn.preprocessing import StandardScaler

scale_cols = ['price', 'discount', 'rating', 'helpful_votes', 'total_votes',
              'effective_price', 'review_word_count', 'helpfulness_ratio',
              'review_year', 'review_month']

scaler = StandardScaler()
scaler.fit(x_train_imputed[scale_cols])

x_train_scaled = x_train_imputed.copy()
x_test_scaled = x_test_imputed.copy()

x_train_scaled[scale_cols] = scaler.transform(x_train_imputed[scale_cols])
x_test_scaled[scale_cols] = scaler.transform(x_test_imputed[scale_cols])

price_std = x_train_scaled['price'].std()
print(f"Standard deviation of 'price' column after scaling: {price_std:.4f}")

Standard deviation of 'price' column after scaling: 1.0000


**Q11.** Using `TfidfVectorizer` with `max_features=500` and `stop_words='english'`, transform the `review_text` column. What is the sum of the TF-IDF values for the very first row of the training set?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=500, stop_words='english')
tfidf.fit(x_train_imputed['review_text'].fillna(''))

x_train_tfidf = tfidf.transform(x_train_imputed['review_text'].fillna('')).toarray()
x_test_tfidf = tfidf.transform(x_test_imputed['review_text'].fillna('')).toarray()

first_row_sum = x_train_tfidf[0].sum()
print(f"Sum of TF-IDF values for the first row of the training set: {first_row_sum:.4f}")

Sum of TF-IDF values for the first row of the training set: 1.8819


**Q12.** Combine the scaled numerical features, one-hot encoded features, and TF-IDF features into a single large matrix using `np.hstack`. What is the total number of columns in this combined training matrix?

In [ ]:
x_train_num_scaled = x_train_scaled[scale_cols].values
x_test_num_scaled = x_test_scaled[scale_cols].values

x_train_combined = np.hstack([x_train_num_scaled, x_train_ohe, x_train_tfidf])
x_test_combined = np.hstack([x_test_num_scaled, x_test_ohe, x_test_tfidf])

total_cols = x_train_combined.shape[1]
print(f"Shape of combined training matrix: {x_train_combined.shape}")
print(f"Total number of columns: {total_cols}")
print(f"Breakdown: {len(scale_cols)} numerical + {x_train_ohe.shape[1]} OHE + {x_train_tfidf.shape[1]} TF-IDF = {total_cols}")

Shape of combined training matrix: (15000, 54)
Total number of columns: 54
Breakdown: 10 numerical + 14 OHE + 30 TF-IDF = 54


**Q13.** **Dimensionality Reduction:** Apply `PCA` (Principal Component Analysis) to the combined matrix from Q12 to reduce the features to `n_components=50`. Use `random_state=42`. What is the total **explained variance ratio** of these 50 components?

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50, random_state=42)
pca.fit(x_train_combined)

x_train_pca = pca.transform(x_train_combined)
x_test_pca = pca.transform(x_test_combined)

total_explained_variance = pca.explained_variance_ratio_.sum()
print(f"Shape of PCA-transformed training matrix: {x_train_pca.shape}")
print(f"Total explained variance ratio of 50 components: {total_explained_variance:.4f}")

Shape of PCA-transformed training matrix: (15000, 50)
Total explained variance ratio of 50 components: 1.0000


**Q14.** **Chi-Square Test:** Use `SelectKBest` with the `chi2` score function to select the top 10 features from the **one-hot encoded categorical features** (from Q9) relative to the `sentiment` target. Which feature was ranked as the most important?

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2

selector = SelectKBest(chi2, k=10)
selector.fit(x_train_ohe, y_train_encoded)

ohe_feature_names = ohe.get_feature_names_out(ohe_cols)
scores = selector.scores_

feature_scores = pd.Series(scores, index=ohe_feature_names).sort_values(ascending=False)
print("Top 10 features by Chi-Square score:")
print(feature_scores.head(10))
print(f"\nMost important feature: {feature_scores.idxmax()}")

Top 10 features by Chi-Square score:
product_category_Books       2.442519
product_category_Beauty      2.341516
user_country_UK              1.896358
user_country_India           1.762422
product_category_Toys        1.593096
user_country_Singapore       1.306386
user_country_USA             1.047612
product_category_Home        0.971391
product_category_Sports      0.697661
product_category_Clothing    0.691296
dtype: float64

Most important feature: product_category_Books


**Q15.** **Tree-Based Importance:** Train a `RandomForestClassifier` (n_estimators=100, random_state=42) on the combined matrix from Q12 (before PCA). List the top 3 features by their `feature_importances_` score.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(x_train_combined, y_train_encoded)

tfidf_feature_names = [f'tfidf_{name}' for name in tfidf.get_feature_names_out()]
all_feature_names = scale_cols + list(ohe_feature_names) + tfidf_feature_names

importances = pd.Series(rf.feature_importances_, index=all_feature_names).sort_values(ascending=False)
print("Top 3 features by feature importance:")
print(importances.head(3))

Top 3 features by feature importance:
rating             0.801112
effective_price    0.021982
price              0.021869
dtype: float64


**Q16.** **Recursive Feature Elimination (RFE):** Using a `LogisticRegression` estimator (max_iter=1000, random_state=42), perform RFE to select the top 20 features from the PCA-transformed data (from Q13). How many features from the original "text" (TF-IDF) space are represented in this top 20 PCA selection?

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator=lr, n_features_to_select=20)
rfe.fit(x_train_pca, y_train_encoded)

selected_components = np.where(rfe.support_)[0]
print(f"Selected PCA components (indices): {selected_components}")

pca_components = pca.components_

tfidf_indices = [i for i, name in enumerate(all_feature_names) if name.startswith('tfidf_')]

threshold = 0.1
tfidf_represented = set()

for comp_idx in selected_components:
    component = pca_components[comp_idx]
    for tfidf_idx in tfidf_indices:
        if abs(component[tfidf_idx]) > threshold:
            tfidf_represented.add(tfidf_idx)

print(f"\nNumber of TF-IDF features represented in top 20 PCA components: {len(tfidf_represented)}")
print(f"TF-IDF features represented: {[all_feature_names[i] for i in tfidf_represented]}")

NameError: name 'x_train_pca' is not defined

**Q17.** Train an `MLPClassifier` with `hidden_layer_sizes=(64, 32)`, `activation='relu'`, `solver='adam'`, and `max_iter=20`. Use the PCA-reduced data from Q13. What is the final training accuracy after 20 iterations?

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam', max_iter=20, random_state=42)
mlp.fit(x_train_pca, y_train_encoded)

train_accuracy = accuracy_score(y_train_encoded, mlp.predict(x_train_pca))
print(f"Final training accuracy after 20 iterations: {train_accuracy:.4f}")

Final training accuracy after 20 iterations: 1.0000


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


**Q18.** **L1 Regularization (Lasso Effect):** Train a `LogisticRegression` model with `penalty='l1'`, `solver='liblinear'`, and `C=0.1` on the combined feature matrix. According to the PPT, L1 makes unimportant features exactly zero. How many feature coefficients in this model are exactly 0?

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_l1 = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, random_state=42)
lr_l1.fit(x_train_combined, y_train_encoded)

zero_coef_count = (lr_l1.coef_ == 0).sum()
print(f"Number of feature coefficients exactly 0: {zero_coef_count}")
print(f"Coefficient matrix shape: {lr_l1.coef_.shape}")
print(f"Total coefficients: {lr_l1.coef_.size}")

Number of feature coefficients exactly 0: 143
Coefficient matrix shape: (3, 54)
Total coefficients: 162


**Q19.** Train the MLP from Q17 again, but this time use `early_stopping=True` and `validation_fraction=0.1`. At which iteration (epoch) did the training stop?

In [ ]:
mlp_early = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
                          max_iter=200, random_state=42, early_stopping=True, validation_fraction=0.1)
mlp_early.fit(x_train_pca, y_train_encoded)

stopping_iteration = mlp_early.n_iter_
print(f"Training stopped at iteration (epoch): {stopping_iteration}")

Training stopped at iteration (epoch): 14


**Q20.** **Binning:** Transform the `rating` column into 3 bins: "Low" (1-2), "Mid" (3), "High" (4-5) using `KBinsDiscretizer` with `encode='ordinal'`. Replace the original `rating` feature with this binned version and retrain the Random Forest from Q15. Compare the Macro F1-scores. Did it improve?


*Read about KBinsDiscretizer here: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.KBinsDiscretizer.html*

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import f1_score

kbd = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform')
kbd.fit(x_train_imputed[['rating']])

x_train_binned = x_train_combined.copy()
x_test_binned = x_test_combined.copy()

rating_idx = scale_cols.index('rating')
x_train_binned[:, rating_idx] = kbd.transform(x_train_imputed[['rating']]).ravel()
x_test_binned[:, rating_idx] = kbd.transform(x_test_imputed[['rating']]).ravel()

y_pred_orig = rf.predict(x_test_combined)
f1_orig = f1_score(y_test_encoded, y_pred_orig, average='macro')

rf_binned = RandomForestClassifier(n_estimators=100, random_state=42)
rf_binned.fit(x_train_binned, y_train_encoded)
y_pred_binned = rf_binned.predict(x_test_binned)
f1_binned = f1_score(y_test_encoded, y_pred_binned, average='macro')

print(f"Macro F1 (original RF):     {f1_orig:.4f}")
print(f"Macro F1 (binned rating RF): {f1_binned:.4f}")
print(f"Improved: {f1_binned > f1_orig}")

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:248: FutureWarning: In version 1.5 onwards, subsample=200_000 will be used by default. Set subsample explicitly to silence this warning in the mean time. Set subsample=None to disable subsampling explicitly.
  warnings.warn(


Macro F1 (original RF):     1.0000
Macro F1 (binned rating RF): 1.0000
Improved: False


**Q21.** Calculate the **Log Loss** of the MLP model from Q17 on the validation set. (Mention up to 4 decimal places).

In [ ]:
from sklearn.metrics import log_loss

y_test_proba = mlp.predict_proba(x_test_pca)
logloss = log_loss(y_test_encoded, y_test_proba)
print(f"Log Loss of MLP model on validation set: {logloss:.4f}")

Log Loss of MLP model on validation set: 0.0003


**Q22.** **Entropy:** For the first instance in the validation set, get the predicted probabilities from the MLP. Calculate the Shannon Entropy ($H = -\sum p_i \log_2 p_i$). Is the model "uncertain" (Entropy > 1.0)?

In [ ]:
first_probs = y_test_proba[0]
entropy = -np.sum(first_probs * np.log2(first_probs + 1e-12))
print(f"Predicted probabilities for first validation instance: {first_probs}")
print(f"Shannon Entropy: {entropy:.4f}")
print(f"Is the model uncertain (Entropy > 1.0)? {entropy > 1.0}")

Predicted probabilities for first validation instance: [1.45313412e-15 1.71219859e-07 9.99999829e-01]
Shannon Entropy: 0.0000
Is the model uncertain (Entropy > 1.0)? False


**Q23.** Load `test.csv`. Apply the **entire pipeline** (Imputation -> Feature Creation -> Scaling -> TF-IDF -> PCA) using only the parameters/fitters learned from the training data. What is the shape of the final test matrix?

In [ ]:
df_test = pd.read_csv('C:\\Users\\Sarthak\\Desktop\\mlp\\test.csv')

test_imputed = df_test.copy()
test_imputed[numerical_cols] = imputer_num.transform(df_test[numerical_cols])
test_imputed[categorical_cols] = imputer_cat.transform(df_test[categorical_cols])

test_imputed['review_date'] = pd.to_datetime(test_imputed['review_date'])
test_imputed['review_year'] = test_imputed['review_date'].dt.year
test_imputed['review_month'] = test_imputed['review_date'].dt.month
test_imputed.drop(columns=['review_date'], inplace=True)

test_imputed['effective_price'] = test_imputed['price'] * (1 - test_imputed['discount'])
test_imputed['effective_price'] = test_imputed['effective_price'].fillna(0)

test_imputed['review_word_count'] = test_imputed['review_text'].fillna('').apply(lambda x: len(x.split()))
test_imputed['is_edited'] = test_imputed['review_text'].apply(lambda x: 1 if '!!!' in str(x) or '???' in str(x) else 0)
test_imputed['helpfulness_ratio'] = test_imputed['helpful_votes'] / (test_imputed['total_votes'] + 1)

test_scaled = test_imputed.copy()
test_scaled[scale_cols] = scaler.transform(test_imputed[scale_cols])

test_ohe = ohe.transform(test_imputed[ohe_cols])

test_tfidf = tfidf.transform(test_imputed['review_text'].fillna('')).toarray()

test_num_scaled = test_scaled[scale_cols].values
test_combined = np.hstack([test_num_scaled, test_ohe, test_tfidf])

test_pca = pca.transform(test_combined)

print(f"Shape of final test matrix (PCA): {test_pca.shape}")
print(f"Shape of combined test matrix (before PCA): {test_combined.shape}")

Shape of final test matrix (PCA): (10000, 50)
Shape of combined test matrix (before PCA): (10000, 54)


**Q24.** Generate predictions for the `test.csv` using the trained MLP. Create a submission dataframe with two columns: `id` and `predicted_sentiment`. What is the frequency of the "positive" class in your test predictions?

In [ ]:
test_predictions_encoded = mlp.predict(test_pca)
predicted_sentiment = le.inverse_transform(test_predictions_encoded)

submission_df = pd.DataFrame({
    'id': df_test['id'],
    'predicted_sentiment': predicted_sentiment
})

positive_freq = (submission_df['predicted_sentiment'] == 'positive').sum()
print(f"Submission dataframe shape: {submission_df.shape}")
print(f"\nFrequency of 'positive' class in test predictions: {positive_freq}")
print(f"\nValue counts:\n{submission_df['predicted_sentiment'].value_counts()}")

Submission dataframe shape: (10000, 2)

Frequency of 'positive' class in test predictions: 5470

Value counts:
predicted_sentiment
positive    5470
negative    2538
neutral     1992
Name: count, dtype: int64


**Q25.** **Consistency Check:** Calculate the Euclidean distance between the mean vector of the training PCA features and the mean vector of the test PCA features. What is the resulting distance?

In [ ]:
train_mean = x_train_pca.mean(axis=0)
test_mean = x_test_pca.mean(axis=0)

euclidean_distance = np.sqrt(np.sum((train_mean - test_mean) ** 2))
print(f"Euclidean distance between mean vectors of train and test PCA features: {euclidean_distance:.4f}")

Euclidean distance between mean vectors of train and test PCA features: 0.0439
